# NB04 — Transformer Causal (Décodeur-only)
> **Phase C & D** : Vaswani et al. 2017, Radford et al. 2019 (GPT-2), Xiong et al. 2020 (Pre-LN).


In [ ]:
# ==============================================================================
# Cellule 1 : Paramètres (Settings) et reproductibilité
# ==============================================================================
import sys
import math
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))
from utils import set_seed, get_device, AminoAcidTokenizer, SequenceDataset, collate_fn, compute_nats_and_perplexity

SEED = 42
set_seed(SEED)
device = get_device()

BATCH_SIZE = 64
EMBED_DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 6
DIM_FEEDFORWARD = 512
DROPOUT = 0.1
LR = 6e-4
WEIGHT_DECAY = 0.01
EPOCHS = 15
GRAD_CLIP = 1.0

print(f"Device: {device} | Seed: {SEED} | Layers: {NUM_LAYERS} | Heads: {NUM_HEADS}")


In [ ]:
# ==============================================================================
# Cellule 2 : Architecture Transformer Causal et Test Unitaire du Masque
# ==============================================================================
class CausalTransformerLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_heads=4, num_layers=6, dim_feedforward=512, max_len=200, dropout=0.1):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True  # Pre-LN
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.head.weight = self.token_embed.weight

    def forward(self, x):
        b, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(b, -1)
        emb = self.token_embed(x) + self.pos_embed(positions)
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
        out = self.transformer(emb, mask=mask, is_causal=True)
        out = self.norm(out)
        return self.head(out)

tokenizer = AminoAcidTokenizer()
model = CausalTransformerLM(tokenizer.vocab_size, EMBED_DIM, NUM_HEADS, NUM_LAYERS, DIM_FEEDFORWARD).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Transformer instancié : {n_params:,} paramètres.")

# Test unitaire : Le futur ne doit pas affecter le passé
model.eval()
with torch.no_grad():
    x1 = torch.tensor([[1, 5, 8, 12]], device=device)
    x2 = torch.tensor([[1, 5, 8, 20]], device=device)  # 4ème token différent
    logits1 = model(x1)
    logits2 = model(x2)
    # Les logits aux positions 0, 1, 2 doivent être strictement identiques
    diff = (logits1[:, :3, :] - logits2[:, :3, :]).abs().max().item()
    print(f"Différence passée : {diff:.6e}")
    assert diff < 1e-5, "Violation de la causalité !"
print("Test unitaire causalité validé avec succès.")


In [ ]:
# ==============================================================================
# Cellule 3 : Entraînement Transformer (15 epochs) et sauvegarde
# ==============================================================================
train_path = ROOT_DIR / "data" / "train.csv"
val_path = ROOT_DIR / "data" / "val.csv"
if not train_path.exists():
    train_path = ROOT_DIR / "data" / "toy" / "train.csv"
    val_path = ROOT_DIR / "data" / "toy" / "val.csv"

train_loader = DataLoader(
    SequenceDataset(pd.read_csv(train_path)["sequence"].tolist(), tokenizer),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, pad_id=tokenizer.pad_id)
)
val_loader = DataLoader(
    SequenceDataset(pd.read_csv(val_path)["sequence"].tolist(), tokenizer),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, pad_id=tokenizer.pad_id)
)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id, reduction='sum')
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

models_dir = ROOT_DIR / "models"
models_dir.mkdir(exist_ok=True)
best_transformer_path = models_dir / "transformer_best.pt"
best_val_nats = float('inf')

print("Début de l'entraînement Transformer...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_tokens = 0.0, 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits.view(-1, tokenizer.vocab_size), targets.view(-1))
        non_pad = (targets != tokenizer.pad_id).sum().item()
        (loss / max(non_pad, 1)).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item()
        total_tokens += non_pad

    train_nats, train_ppl = compute_nats_and_perplexity(total_loss, total_tokens)
    model.eval()
    val_loss, val_tokens = 0.0, 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            loss = criterion(logits.view(-1, tokenizer.vocab_size), targets.view(-1))
            val_loss += loss.item()
            val_tokens += (targets != tokenizer.pad_id).sum().item()
    val_nats, val_ppl = compute_nats_and_perplexity(val_loss, val_tokens)

    if val_nats < best_val_nats:
        best_val_nats = val_nats
        torch.save(model.state_dict(), best_transformer_path)
        star = "★"
    else:
        star = ""

    print(f"Epoch {epoch:2d}/{EPOCHS} | Train nats: {train_nats:.3f} | Val nats: {val_nats:.3f} (ppl: {val_ppl:.1f}) {star}")

print(f"Terminé. Meilleur modèle sauvegardé dans {best_transformer_path}")
